In [ ]:
import pandas as pd 
import numpy as np


In [ ]:
def code_choice(row):
    row = str(row).lower()  # str() guards against any NaN/non-string cell
    if 'not' in row: 
        return 0
    else: 
        return 1


In [ ]:
cambrian8b = pd.read_csv("vlm_baselines/cambrian-8b-annotations.csv")
# resample on sentiment_2, 250 in each class
# Version-robust per-group sampling (pandas>=2.2 excludes the grouping
# column from groupby.apply; this yields identical rows/results).
cambrian8b = pd.concat(
    [g.sample(250, random_state=777, replace=False) for _, g in cambrian8b.groupby('sentiment_2')]
).reset_index(drop=True)
cambrian8b['gt'] = cambrian8b['choice'].apply(code_choice)
print(cambrian8b['gt'].mean())
cambrian8b = cambrian8b[['sentiment_2', 'gt']]
cambrian8b['correct'] = cambrian8b['sentiment_2'] == cambrian8b['gt']
print(cambrian8b['correct'].mean())

# split into _1 and _0 on classification 
cambrian8b_1 = cambrian8b[cambrian8b['sentiment_2'] == 1]
cambrian8b_0 = cambrian8b[cambrian8b['sentiment_2'] == 0]
print(len(cambrian8b_1), len(cambrian8b_0))

# compute p(correct) for each split
print(cambrian8b_1['correct'].mean(), cambrian8b_0['correct'].mean())

assert len(cambrian8b_1) == len(cambrian8b_0)


In [ ]:
cambrian13b = pd.read_csv('../../data/processed/inspection_set.csv')

def code_inspection_set_choice(row):
    # The 1 unannotated row (NaN choice) is excluded from metrics: return NaN
    # so it can be dropped before computing PPV/FOR (committed table values
    # were computed on annotated rows only).
    if pd.isna(row):
        return np.nan
    row = str(row).lower()
    if 'drivable' in row:
        return 0
    else:
        return 1

cambrian13b['gt'] = cambrian13b['choice'].apply(code_inspection_set_choice)
# Exclude the unannotated (NaN choice) row from the metric computation.
cambrian13b = cambrian13b.dropna(subset=['gt'])
print(cambrian13b['gt'].mean())
cambrian13b = cambrian13b[['sentiment_1', 'gt']]
cambrian13b['correct'] = cambrian13b['sentiment_1'] == cambrian13b['gt']
print(cambrian13b['correct'].mean())

# split into _1 and _0 on classification
cambrian13b_1 = cambrian13b[cambrian13b['sentiment_1'] == 1]
cambrian13b_0 = cambrian13b[cambrian13b['sentiment_1'] == 0]
print(len(cambrian13b_1), len(cambrian13b_0))

# compute p(correct) for each split
print(cambrian13b_1['correct'].mean(), cambrian13b_0['correct'].mean())

# Splits may differ by the single excluded unannotated (NaN choice) row.
assert abs(len(cambrian13b_1) - len(cambrian13b_0)) <= 1


In [ ]:
janus = pd.read_csv("vlm_baselines/januspro_onefoot_annotations.csv")
janus['answer'] = janus['answer'].apply(lambda x: 1 if 'yes' in x.lower() else 0)
# resample on answer, 250 in each class
# Version-robust per-group sampling (pandas>=2.2 excludes the grouping
# column from groupby.apply; this yields identical rows/results).
janus = pd.concat(
    [g.sample(250, random_state=777, replace=False) for _, g in janus.groupby('answer')]
).reset_index(drop=True)
janus['gt'] = janus['choice'].apply(code_choice)
print(janus['gt'].mean())
janus = janus[['answer', 'gt']]
# code answer 
# if answer contains yes, 1. else 0

janus['correct'] = janus['answer'] == janus['gt']
print(janus['correct'].mean())

# split into _1 and _0 on classification
janus_1 = janus[janus['answer'] == 1]
janus_0 = janus[janus['answer'] == 0]

print(len(janus_1), len(janus_0))

# compute p(correct) for each split
print(janus_1['correct'].mean(), janus_0['correct'].mean())

assert len(janus_1) == len(janus_0)

In [ ]:
clip = pd.read_csv("vlm_baselines/clip-vitg-annotations.csv")
clip['positive'] = clip['positive'].astype(int)

# resample on positive, 250 in each class
# Version-robust per-group sampling (pandas>=2.2 excludes the grouping
# column from groupby.apply; this yields identical rows/results).
clip = pd.concat(
    [g.sample(250, random_state=777, replace=False) for _, g in clip.groupby('positive')]
).reset_index(drop=True)

clip['gt'] = clip['choice'].apply(code_choice)
print(clip['gt'].mean())


clip = clip[['positive', 'gt']]
clip['correct'] = clip['positive'] == clip['gt']
print(clip['correct'].mean())

# split into _1 and _0 on classification
clip_1 = clip[clip['positive'] == 1]
clip_0 = clip[clip['positive'] == 0]

print(len(clip_1), len(clip_0))

# compute p(correct) for each split
print(clip_1['correct'].mean(), clip_0['correct'].mean())

assert len(clip_1) == len(clip_0)


In [ ]:
supervised = pd.read_csv("vlm_baselines/supervised_annotations.csv")
# resample on pred_label, 250 in each class
# Version-robust per-group sampling (pandas>=2.2 excludes the grouping
# column from groupby.apply; this yields identical rows/results).
supervised = pd.concat(
    [g.sample(250, random_state=777, replace=False) for _, g in supervised.groupby('pred_label')]
).reset_index(drop=True)

supervised['gt'] = supervised['choice'].apply(code_choice)
print(supervised['gt'].mean())

supervised = supervised[['pred_label', 'gt']]
supervised['correct'] = supervised['pred_label'] == supervised['gt']
print(supervised['correct'].mean())

# split into _1 and _0 on classification
supervised_1 = supervised[supervised['pred_label'] == 1]
supervised_0 = supervised[supervised['pred_label'] == 0]

print(len(supervised_1), len(supervised_0))

# compute p(correct) for each split
print(supervised_1['correct'].mean(), supervised_0['correct'].mean())

assert len(supervised_1) == len(supervised_0)



In [ ]:
# run a t test between cambrian13b and each other baseline 
# test the _1 and _0 separately
from scipy.stats import ttest_ind

positive_baseline_names = ['cambrian8b_1', 'janus_1', 'clip_1', 'supervised_1']
negative_baseline_names = ['cambrian8b_0', 'janus_0', 'clip_0', 'supervised_0']
positive_baselines = [cambrian8b_1, janus_1, clip_1, supervised_1]
negative_baselines = [cambrian8b_0, janus_0, clip_0, supervised_0]

for i, baseline in enumerate(positive_baselines):
    # run t test with cambrian13b_1 and baseline
    t, p = ttest_ind(cambrian13b_1['correct'], baseline['correct'])
    print(f"{positive_baseline_names[i]}: t = {t}, p = {p}")
    print(f"{positive_baseline_names[i]}: p(correct) = {baseline['correct'].mean()}")
    print()

for i, baseline in enumerate(negative_baselines):
    # run t test with cambrian13b_0 and baseline

    t, p = ttest_ind(cambrian13b_0['correct'], baseline['correct'])
    print(f"{negative_baseline_names[i]}: t = {t}, p = {p}")
    print(f"{negative_baseline_names[i]}: p(correct) = {baseline['correct'].mean()}")
    print()







